In [ ]:
!pip install pandas matplotlib seaborn numpy

# Scentence 개인화 지수 (Graph 기반) 테스트 결과 시각화

이 노트북은 개인화 테스트 결과를 시각화합니다.

In [ ]:
# 필요한 패키지 설치
# 아래 명령어를 터미널에서 실행하거나, 주석을 해제하여 셀에서 실행하세요
# !pip install pandas matplotlib seaborn numpy

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from datetime import datetime
from collections import Counter

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
sns.set_style("whitegrid")
plt.style.use('seaborn-v0_8-darkgrid')

## 1. 데이터 로드

CSV 파일 경로를 지정하세요:

In [ ]:
# CSV 파일 경로 (실제 파일명으로 변경하세요)
CSV_PATH = "backend/tests/evaluation_results/personalization_graph_details_20260205_122857.csv"

# 데이터 로드
df = pd.read_csv(CSV_PATH)

print(f"총 테스트 수: {len(df)}")
print(f"\n컬럼: {list(df.columns)}")
df.head()

## 2. 브랜드 다양성 지표 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 고유 브랜드 수 분포
axes[0, 0].hist(df['unique_brands'], bins=range(1, df['unique_brands'].max()+2), 
                color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('고유 브랜드 수')
axes[0, 0].set_ylabel('빈도')
axes[0, 0].set_title('쿼리별 고유 브랜드 수 분포')
axes[0, 0].axvline(df['unique_brands'].mean(), color='red', linestyle='--', 
                   label=f'평균: {df["unique_brands"].mean():.2f}')
axes[0, 0].legend()

# 브랜드 엔트로피 분포
axes[0, 1].hist(df['brand_entropy'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('브랜드 엔트로피')
axes[0, 1].set_ylabel('빈도')
axes[0, 1].set_title('브랜드 엔트로피 분포')
axes[0, 1].axvline(df['brand_entropy'].mean(), color='red', linestyle='--',
                   label=f'평균: {df["brand_entropy"].mean():.4f}')
axes[0, 1].legend()

# 최다 브랜드 비율 분포
axes[1, 0].hist(df['top_brand_ratio'], bins=20, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('최다 브랜드 비율')
axes[1, 0].set_ylabel('빈도')
axes[1, 0].set_title('최다 브랜드 비율 분포')
axes[1, 0].axvline(df['top_brand_ratio'].mean(), color='red', linestyle='--',
                   label=f'평균: {df["top_brand_ratio"].mean():.4f}')
axes[1, 0].legend()

# 지니계수 분포
axes[1, 1].hist(df['gini_coefficient'], bins=30, color='gold', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('지니계수')
axes[1, 1].set_ylabel('빈도')
axes[1, 1].set_title('지니계수 분포')
axes[1, 1].axvline(df['gini_coefficient'].mean(), color='red', linestyle='--',
                   label=f'평균: {df["gini_coefficient"].mean():.4f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. 종합 다양성 점수 분석

In [ ]:
# 다양성 점수 계산 (테스트와 동일한 공식)
def calculate_diversity_score(row):
    entropy_score = min(row['brand_entropy'] / 5.0, 1.0) * 40
    top_brand_score = (1 - row['top_brand_ratio']) * 30
    gini_score = (1 - row['gini_coefficient']) * 30
    return entropy_score + top_brand_score + gini_score

df['diversity_score'] = df.apply(calculate_diversity_score, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 다양성 점수 분포
axes[0].hist(df['diversity_score'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('다양성 점수')
axes[0].set_ylabel('빈도')
axes[0].set_title('종합 다양성 점수 분포')
axes[0].axvline(df['diversity_score'].mean(), color='red', linestyle='--',
                label=f'평균: {df["diversity_score"].mean():.1f}')
axes[0].axvline(80, color='green', linestyle=':', alpha=0.5, label='우수 (80+)')
axes[0].axvline(60, color='orange', linestyle=':', alpha=0.5, label='양호 (60+)')
axes[0].axvline(40, color='red', linestyle=':', alpha=0.5, label='주의 (40+)')
axes[0].legend()

# 다양성 등급별 분포
def get_diversity_grade(score):
    if score >= 80:
        return '우수'
    elif score >= 60:
        return '양호'
    elif score >= 40:
        return '주의'
    else:
        return '위험'

df['diversity_grade'] = df['diversity_score'].apply(get_diversity_grade)
grade_counts = df['diversity_grade'].value_counts()

grade_order = ['우수', '양호', '주의', '위험']
grade_counts = grade_counts.reindex(grade_order, fill_value=0)

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
axes[1].bar(grade_counts.index, grade_counts.values, color=colors)
axes[1].set_xlabel('다양성 등급')
axes[1].set_ylabel('쿼리 수')
axes[1].set_title('다양성 등급별 분포')

for i, v in enumerate(grade_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n평균 다양성 점수: {df['diversity_score'].mean():.1f}/100")
print(f"다양성 등급 분포:")
for grade in grade_order:
    count = grade_counts[grade]
    pct = count / len(df) * 100
    print(f"  {grade}: {count}개 ({pct:.1f}%)")

## 4. 지표 간 상관관계 분석

In [ ]:
# 상관관계 히트맵
corr_cols = ['unique_brands', 'brand_entropy', 'top_brand_ratio', 'gini_coefficient', 'diversity_score']
corr_matrix = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 히트맵
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0, 
            square=True, ax=axes[0], fmt='.3f')
axes[0].set_title('지표 간 상관관계')

# 산점도 매트릭스 (주요 지표만)
sample_df = df.sample(min(100, len(df)))
axes[1].scatter(sample_df['brand_entropy'], sample_df['top_brand_ratio'], 
                c=sample_df['diversity_score'], cmap='RdYlGn', alpha=0.6, s=50)
axes[1].set_xlabel('브랜드 엔트로피')
axes[1].set_ylabel('최다 브랜드 비율')
axes[1].set_title('엔트로피 vs 최다 브랜드 비율')
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('다양성 점수')

plt.tight_layout()
plt.show()

## 5. 쿼리 유형별 분석

In [ ]:
# 쿼리에서 성별 추출
def extract_gender_from_query(query):
    query_lower = query.lower()
    male_keywords = ['남자', '남성', '남친', '남자친구', '남편', '아빠', '아버지']
    female_keywords = ['여자', '여성', '여친', '여자친구', '아내', '엄마', '어머니']
    
    for kw in male_keywords:
        if kw in query_lower:
            return '남성'
    for kw in female_keywords:
        if kw in query_lower:
            return '여성'
    return '중성/미지정'

df['gender_category'] = df['query'].apply(extract_gender_from_query)

# 성별별 다양성 점수
gender_stats = df.groupby('gender_category').agg({
    'diversity_score': ['mean', 'std', 'count'],
    'unique_brands': 'mean',
    'brand_entropy': 'mean'
}).round(2)

print("성별별 다양성 통계:")
print(gender_stats)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

gender_scores = df.groupby('gender_category')['diversity_score'].mean()
gender_scores.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_xlabel('성별')
axes[0].set_ylabel('평균 다양성 점수')
axes[0].set_title('성별별 평균 다양성 점수')
axes[0].tick_params(axis='x', rotation=0)
axes[0].axhline(60, color='orange', linestyle='--', alpha=0.5, label='양호 기준')
axes[0].legend()

# 박스플롯
df.boxplot(column='diversity_score', by='gender_category', ax=axes[1])
axes[1].set_xlabel('성별')
axes[1].set_ylabel('다양성 점수')
axes[1].set_title('성별별 다양성 점수 분포')
plt.suptitle('')  # 자동 생성된 제목 제거

plt.tight_layout()
plt.show()

## 6. 브랜드 분포 분석

In [ ]:
# 모든 브랜드 집계
all_brands = Counter()
for brands_json in df['brands']:
    brands_dict = json.loads(brands_json)
    all_brands.update(brands_dict)

print(f"총 고유 브랜드 수: {len(all_brands)}")
print(f"\n상위 15개 브랜드:")
for brand, count in all_brands.most_common(15):
    pct = count / df['total_recommendations'].sum() * 100
    print(f"  {brand}: {count}회 ({pct:.1f}%)")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 상위 브랜드 막대 그래프
top_brands = dict(all_brands.most_common(15))
axes[0].barh(list(top_brands.keys())[::-1], list(top_brands.values())[::-1], 
           color='steelblue')
axes[0].set_xlabel('추천 횟수')
axes[0].set_title('상위 15개 브랜드 추천 횟수')

# 브랜드 분포 파이 차트 (상위 10개 + 기타)
top_10 = dict(all_brands.most_common(10))
others_count = sum(all_brands.values()) - sum(top_10.values())
if others_count > 0:
    top_10['기타'] = others_count

axes[1].pie(top_10.values(), labels=top_10.keys(), autopct='%1.1f%%', 
           startangle=90)
axes[1].set_title('브랜드별 추천 비율 (상위 10개 + 기타)')

plt.tight_layout()
plt.show()

## 7. 상위/하위 성능 쿼리 분석

In [ ]:
# 다양성 점수가 높은/낮은 쿼리 분석
print("=== 다양성 점수 상위 10개 쿼리 ===")
high_diversity = df.nlargest(10, 'diversity_score')[['query', 'unique_brands', 'brand_entropy', 'diversity_score']]
for _, row in high_diversity.iterrows():
    print(f"\n{row['query'][:50]}...")
    print(f"  고유 브랜드: {row['unique_brands']}, 엔트로피: {row['brand_entropy']:.3f}, 점수: {row['diversity_score']:.1f}")

print("\n\n=== 다양성 점수 하위 10개 쿼리 ===")
low_diversity = df.nsmallest(10, 'diversity_score')[['query', 'unique_brands', 'brand_entropy', 'diversity_score']]
for _, row in low_diversity.iterrows():
    print(f"\n{row['query'][:50]}...")
    print(f"  고유 브랜드: {row['unique_brands']}, 엔트로피: {row['brand_entropy']:.3f}, 점수: {row['diversity_score']:.1f}")

## 8. 성능 지표 요약

In [ ]:
# 성능 지표 계산
metrics = {
    '총 테스트 쿼리 수': len(df),
    '평균 고유 브랜드 수': f"{df['unique_brands'].mean():.2f}",
    '평균 브랜드 엔트로피': f"{df['brand_entropy'].mean():.4f}",
    '평균 최다 브랜드 비율': f"{df['top_brand_ratio'].mean():.4f}",
    '평균 지니계수': f"{df['gini_coefficient'].mean():.4f}",
    '종합 다양성 점수': f"{df['diversity_score'].mean():.1f}/100",
    '우수 등급 비율': f"{(df['diversity_score'] >= 80).mean() * 100:.1f}%",
    '양호 이상 비율': f"{(df['diversity_score'] >= 60).mean() * 100:.1f}%",
}

print("=== 개인화 지수 테스트 결과 요약 ===")
for key, value in metrics.items():
    print(f"{key}: {value}")

# 해석
mean_score = df['diversity_score'].mean()
if mean_score >= 80:
    interpretation = "우수: 매우 다양한 브랜드가 추천되고 있습니다"
elif mean_score >= 60:
    interpretation = "양호: 적절한 브랜드 다양성을 보여줍니다"
elif mean_score >= 40:
    interpretation = "주의: 일부 브랜드로 편향된 경향이 있습니다"
else:
    interpretation = "위험: 특정 브랜드에 심하게 편중되어 있습니다"

print(f"\n해석: {interpretation}")

## 9. 결과 저장

시각화 결과를 저장하려면 아래 코드를 실행하세요:

In [ ]:
# 그래프 저장
# plt.savefig('personalization_test_visualization.png', dpi=300, bbox_inches='tight')
# print("그래프 저장 완료: personalization_test_visualization.png")